### Gradient Boosting Intuition (Regression)

We start with a base estimator.

- In regression, the base prediction is usually the **mean of target values**

Example (Package):

y = [4.5, 11.0, 6.0, 8.0]
mean = 7.5


---

### Stage 1

- Initial prediction for all data points = **7.5**

- Compute residuals (error):

$$
\text{Residual} = y - \hat{y}
$$

---

### Stage 2

- Train a **decision tree**:
  - Input → same features (CGPA)  
  - Output → residuals from Stage 1  

- This tree learns how to correct the initial prediction  

- Combine model:

$$
\text{New Prediction} = \text{Base Model} + \text{Tree 1}
$$

---

### Stage 3

- Compute new residuals:
  - error of combined model  

- Train another decision tree:
  - Input → same features  
  - Output → new residuals  

- Update model:

$$
\text{Prediction} = \text{Previous Model} + \text{Tree 2}
$$

---

### Final Idea

- Keep adding trees sequentially  
- Each tree learns from **previous errors**  

This is how regression works in **Gradient Boosting**

-----
----

### XGBoost vs Gradient Boosting

- XGBoost works in a similar way as Gradient Boosting  
- The overall flow remains the same:
  - start with a base prediction  
  - compute residuals  
  - train trees sequentially to correct errors  

- The major difference lies in the **decision trees**

- In Gradient Boosting:
  - uses standard (vanilla) decision trees  
  - based on criteria like Gini / Entropy  

- In XGBoost:
  - uses a different type of decision tree  
  - tree construction is more optimized and mathematically driven  

---

### Key Idea

- Flow is same as Gradient Boosting  
- Main twist → **how trees are built in XGBoost**

---
---

# Step by step calculation

| CGPA | Package |
|------|---------|
| 6.7  | 4.5     |
| 9.0  | 11.0    |
| 7.5  | 6.0     |
| 5.0  | 8.0     |

### Stage 1

- Initial prediction (mean):

y = [4.5, 11.0, 6.0, 8.0]
mean ≈ 7.3


- Model 1 output for all points = **7.3**
  
### Residuals (res₁)

- Create a new dataset:

CGPA | residual₁
6.7 | -2.8
9.0 | 3.7
7.5 | -1.3
5.0 | 0.7

- Residual:

$$
r_i = y_i - \hat{y}
$$

---

### Similarity Score (SS)

- All residuals are placed in one leaf node

$$
SS = \frac{(\sum r_i)^2}{n + \lambda}
$$

---

### Splitting the Data

- Sort CGPA:

  - 5.0
  - 6.7 → split at 5.85
  - 7.5 → split at 7.1
  - 9.0 → split at 8.25


- For each split:
  - compute similarity score  
  - choose the split with maximum gain  
## key idea
- Start with mean prediction  
- Compute residuals  
- Find best split using similarity score  

| CGPA | Package | Model 1 (Prediction) | Residual₁ |
|------|---------|---------------------|-----------|
| 6.7  | 4.5     | 7.3                 | -2.8      |
| 9.0  | 11.0    | 7.3                 | 3.7       |
| 7.5  | 6.0     | 7.3                 | -1.3      |
| 5.0  | 8.0     | 7.3                 | 0.7       |

### Split Evaluation (Stage 2 Tree Building)

#### Split 1: CGPA < 5.85

cgpa < 5.85
|
|-- yes → [0.7]
|
|-- no → [-2.8, 3.7, -1.3]



- SS_left = 0.49  
- SS_right ≈ 0.05  

- Gain:
$$
Gain = SS_{left} + SS_{right} - SS_{root}
$$

$$
= 0.49 + 0.05 - 0.02 = 0.52
$$

---

#### Split 2: CGPA < 7.1
cgpa < 7.1
|
|-- yes → [0.7, -2.8]
|
|-- no → [-1.3, 3.7]



- SS_left = 2.20  
- SS_right = 2.88  

- Gain:
$$
= 2.20 + 2.88 - 0.02 = 5.06
$$

- Better than previous split  

---

#### Split 3: CGPA < 8.25
cgpa < 8.25
|
|-- yes → [0.7, -2.8, -1.3]
|
|-- no → [3.7]



- SS_left = 3.85  
- SS_right = 13.69  

- Gain:
$$
= 3.85 + 13.69 - 0.02 = 17.52
$$

---

### Final Decision

- Highest gain = **17.52**  
- Best split = **CGPA < 8.25**

---

### Stage 2 Tree

cgpa < 8.25
|
|-- yes → [0.7, -2.8, -1.3]
|
|-- no → [3.7]

---
---

### Further Splitting After Root Node

Root split:

cgpa < 8.25
|
|-- yes → [0.7, -2.8, -1.3]
|-- no → [3.7]


---

### Option 1: Split at CGPA < 5.85

cgpa < 8.25
|
|-- yes → split (cgpa < 5.85)
| |
| |-- yes → [0.7]
| |
| |-- no → [-2.8, -1.3]
|
|-- no → [3.7]



- SS (parent) ≈ 0.02  
- SS (left split) ≈ 3.65  


### Option 2: Split at CGPA < 7.1
cgpa < 8.25
|
|-- yes → split (cgpa < 7.1)
| |
| |-- yes → [0.7, -2.8]
| |
| |-- no → [-1.3]
|
|-- no → [3.7]


- SS (parent) ≈ 0.02  
- SS (left split) ≈ 2.85  



### Key Idea

- After selecting the best root split, we continue splitting child nodes  
- At each step:
  - try different split points  
  - compute similarity score  
  - choose the split with maximum gain  

---
---

### Comparing Further Splits (Gain Calculation)

#### Option 1: Split at CGPA < 5.85

[0.7, -2.8, -1.3]
|
|-- yes → [0.7]
|
|-- no → [-2.8, -1.3]



- SS_left:
$$
SS_L = \frac{(0.7)^2}{1} = 0.49
$$

- SS_right:
$$
SS_R = \frac{(-2.8 + -1.3)^2}{2} = \frac{(-4.1)^2}{2} = \frac{16.81}{2} = 8.40
$$

- Gain:
$$
Gain = SS_L + SS_R - SS_{parent}
$$

$$
= 0.49 + 8.40 - 3.85 = 5.04
$$

---

#### Option 2: Split at CGPA < 7.1


[0.7, -2.8, -1.3]
|
|-- yes → [0.7, -2.8]
|
|-- no → [-1.3]



- SS_left:
$$
SS_L = \frac{(0.7 + -2.8)^2}{2} = \frac{(-2.1)^2}{2} = \frac{4.41}{2} = 2.20
$$

- SS_right:
$$
SS_R = \frac{(-1.3)^2}{1} = 1.69
$$

- Gain:
$$
= 2.20 + 1.69 - 3.85 = 0.04
$$

---

### Final Decision

- Gain (5.85 split) = **5.04**  
- Gain (7.1 split) = **0.04**

- Best split = **CGPA < 5.85**


### Final Tree Structure [stage 2]

- Root: cgpa < 8.25  
  - Left:
    - cgpa < 5.85  
      - Left → [0.7]  
      - Right → [-2.8, -1.3]  
  - Right → [3.7]

> ### In Xgboost the default depth_tree is 6 but in our case it will be overfitting 6 is for the bigger data 

---
---

### Leaf Output Value (XGBoost)

- Output value of a leaf node:

$$
\text{Leaf Value} = \frac{\sum \text{residual}}{n + \lambda}
$$

- where:
  - $$ n $$ = number of residuals in the leaf  
  - $$ \lambda $$ = regularization parameter  

---

- In our case:
  - $$ \lambda = 0 $$

So,

$$
\text{Leaf Value} = \frac{\sum \text{residual}}{n}
$$

---

### Key Idea

- Each leaf stores a value that represents the **correction to previous prediction**  
- These values are added to improve the model step by step  

---
---

in our case 
- In our case:
  - $$ \lambda = 0 $$

> So the output of each node will be average

---
---

> ## Model 2  output

- 7.37 + lr * output of decision tree in stage 2 
- default value of lr is 0.3

### Final Tree Structure [stage 2]

- Root: cgpa < 8.25  
  - Left:
    - cgpa < 5.85  
      - Left → [0.7]  
      - Right → [-2.8, -1.3]  
  - Right → [3.7]

-----

### Model Update (Stage-wise)

| CGPA | Package | Model 1 | Residual₁ | Model 2                          | Residual₂ |
|------|---------|---------|------------|-----------------------------------------------|------------|
| 6.7  | 4.5     | 7.3     | -2.8       | $$7.3 + 0.3 \times (-2.05) = 6.69$$           | -2.19      |
| 9.0  | 11.0    | 7.3     | 3.7        | $$7.3 + 0.3 \times 3.7 = 8.41$$               | 2.59       |
| 7.5  | 6.0     | 7.3     | -1.3       | $$7.3 + 0.3 \times (-2.05) = 6.69$$           | -0.69      |
| 5.0  | 8.0     | 7.3     | 0.7        | $$7.3 + 0.3 \times 0.7 = 7.51$$               | 0.49       |

----
---


> Goal is to make the residual zero

---

### Stage 3 (Next Model Update)

- We train a new decision tree (**DT₂**) on residuals from the previous stage  

- Final prediction becomes:

$$
\hat{y} = \text{mean} + \eta \cdot DT_1 + \eta \cdot DT_2
$$

- where:
  - $$ \eta $$ = learning rate  
  - $$ DT_1, DT_2 $$ = outputs from decision trees  

---

### Key Idea

- Each new tree learns the **remaining error**  
- Predictions are updated by adding scaled outputs of trees  
- Process continues until error is minimized  

---
---

### Stage 3: Building DT₂ using Residual₂

- New dataset:
CGPA | residual₂


- We repeat the same process:
  - compute similarity score (SS)  
  - try different split points  
  - calculate gain  
  - choose best split  

---

### Step 1: Sort CGPA
5.0, 6.7, 7.5, 9.0


- Candidate splits:

5.85, 7.1, 8.25


---

### Step 2: Evaluate Splits

- For each split:
  - divide residual₂ into left and right  
  - compute:

$$
SS = \frac{(\sum r)^2}{n + \lambda}
$$

- then compute gain:

$$
Gain = SS_{left} + SS_{right} - SS_{root}
$$

---

### Step 3: Select Best Split

- Choose split with **maximum gain**  
- That becomes the root of **DT₂**

---

### Final Idea

- DT₂ is built using **residual₂**  
- Same process as DT₁  
- Each stage keeps reducing error  

- Updated model:

$$
\hat{y} = \text{mean} + \eta \cdot DT_1 + \eta \cdot DT_2
$$

---
---

### Multiple Features in XGBoost

- When there are multiple features:
  - XGBoost checks **all features** for splitting  
  - For each feature:
    - tries all possible split points  
    - computes gain  
  - Final split is chosen based on **maximum gain (best feature + best threshold)**  

---

### Numerical and Categorical Features

- Numerical features:
  - handled directly  
  - example: cgpa < 7.5  

---

- Categorical features:

  - Binary categorical:
    - convert into 0 / 1  
    - example: yes = 1, no = 0  
    - split happens like: feature < 0.5  

  - Multi-class categorical:
    - needs encoding  

    - Label encoding:
      - CSE → 0, ECE → 1, ME → 2  

    - One-hot encoding:
      - CSE → [1,0,0]  
      - ECE → [0,1,0]  
      - ME  → [0,0,1]  

---

### Important

- XGBoost (by default):
  - does not directly handle categorical features  
  - requires encoding before training  

---

### Key Idea

- Try splits across all features  
- Select the best split globally  
- Same tree logic applies after encoding  

- Final split = best feature + best split point based on gain  

---
---

>
> when you have small dataset : you use algorithm 1 [exact greedy]
on large dataset we use 2nd algo .
Refer : 

Refer : [XGBoost_orginal_paper_page_3](https://arxiv.org/pdf/1603.02754)